In [13]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

print("📂 Outputs détecté :", OUT_DIR.resolve())


📂 Outputs détecté : C:\Users\Malaka\Desktop\insightia_nlp\outputs


In [14]:
files = sorted(OUT_DIR.glob("block4_zoom_motif_situation*"))

print("📄 Fichiers trouvés :")
for f in files:
    print(" -", f.name)

if not files:
    raise FileNotFoundError(
        "❌ Aucun fichier block4_zoom_motif_situation* trouvé dans outputs/"
    )

path = files[0]
print("✅ Fichier utilisé :", path.name)


📄 Fichiers trouvés :
 - block4_zoom_motif_situation.csv
✅ Fichier utilisé : block4_zoom_motif_situation.csv


In [15]:
ext = path.suffix.lower()

if ext in [".csv", ".txt"]:
    df = pd.read_csv(path)
elif ext in [".xlsx", ".xls"]:
    df = pd.read_excel(path)
elif ext == ".ods":
    df = pd.read_excel(path, engine="odf")
else:
    print("⚠️ Extension inconnue, tentative CSV")
    df = pd.read_csv(path)

print("✅ Données chargées")
print("Colonnes :", list(df.columns))
df.head()


✅ Données chargées
Colonnes : ['motif', 'situation', 'n_recent', 'n_prev', 'delta', 'growth_ratio', 'recent_months', 'prev_months']


,motif,situation,n_recent,n_prev,delta,growth_ratio,recent_months,prev_months
0,bug,essais_multiples,49,36,13,1.351351,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01"
1,parcours,blocage_parcours,91,71,20,1.277778,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01"
2,bug,recours_support,24,19,5,1.250000,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01"
3,authentification,perte_temps,44,36,8,1.216216,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01"
4,authentification,blocage_parcours,22,20,2,1.095238,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01"


In [16]:
def pick_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

col_volume = pick_column(df, [
    "n_recent", "recent_n", "count_recent", "volume_recent", "n"
])

col_growth = pick_column(df, [
    "growth_ratio", "growth", "ratio", "trend_ratio"
])

col_delta = pick_column(df, [
    "delta", "diff", "change"
])

print("📊 Colonnes détectées :")
print(" - Volume :", col_volume)
print(" - Growth :", col_growth)
print(" - Delta  :", col_delta)

if col_volume is None:
    raise ValueError("❌ Aucune colonne de volume trouvée")


📊 Colonnes détectées :
 - Volume : n_recent
 - Growth : growth_ratio
 - Delta  : delta


In [17]:
volume = df[col_volume].fillna(0)

if col_growth is not None:
    growth = df[col_growth].fillna(1)
elif col_delta is not None:
    growth = (df[col_delta].fillna(0) / volume.replace(0, 1)) + 1
else:
    growth = 1

df["priority_score"] = volume * growth

df_sorted = df.sort_values("priority_score", ascending=False)

print("✅ Score calculé")
df_sorted.head(10)


✅ Score calculé


,motif,situation,n_recent,n_prev,delta,growth_ratio,recent_months,prev_months,priority_score
10,photo,blocage_parcours,267,273,-6,0.978102,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",261.153285
5,photo,perte_temps,168,154,14,1.090323,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",183.174194
16,photo,essais_multiples,137,159,-22,0.862500,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",118.162500
1,parcours,blocage_parcours,91,71,20,1.277778,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",116.277778
12,parcours,perte_temps,109,117,-8,0.932203,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",101.610169
13,parcours,essais_multiples,112,124,-12,0.904000,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",101.248000
8,lenteur,perte_temps,73,72,1,1.013699,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",74.000000
0,bug,essais_multiples,49,36,13,1.351351,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",66.216216
9,lenteur,essais_multiples,66,66,0,1.000000,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",66.000000
18,bug,blocage_parcours,66,79,-13,0.837500,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",55.275000


In [18]:
keep_cols = [
    c for c in [
        "motif",
        "situation",
        col_volume,
        col_growth,
        col_delta,
        "priority_score",
        "recent_months",
        "prev_months"
    ]
    if c and c in df_sorted.columns
]

backlog = df_sorted[keep_cols].copy()

if "motif" in backlog.columns and "situation" in backlog.columns:
    backlog["problem_statement"] = (
        "Problème : " + backlog["motif"].astype(str)
        + " — situation : " + backlog["situation"].astype(str)
    )

backlog.head(10)


,motif,situation,n_recent,growth_ratio,delta,priority_score,recent_months,prev_months,problem_statement
10,photo,blocage_parcours,267,0.978102,-6,261.153285,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : photo — situation : blocage_parcours
5,photo,perte_temps,168,1.090323,14,183.174194,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : photo — situation : perte_temps
16,photo,essais_multiples,137,0.862500,-22,118.162500,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : photo — situation : essais_multiples
1,parcours,blocage_parcours,91,1.277778,20,116.277778,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : parcours — situation : blocage_parc...
12,parcours,perte_temps,109,0.932203,-8,101.610169,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : parcours — situation : perte_temps
13,parcours,essais_multiples,112,0.904000,-12,101.248000,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : parcours — situation : essais_multi...
8,lenteur,perte_temps,73,1.013699,1,74.000000,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : lenteur — situation : perte_temps
0,bug,essais_multiples,49,1.351351,13,66.216216,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : bug — situation : essais_multiples
9,lenteur,essais_multiples,66,1.000000,0,66.000000,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : lenteur — situation : essais_multiples
18,bug,blocage_parcours,66,0.837500,-13,55.275000,"2024-10-01, 2024-11-01, 2024-12-01","2024-07-01, 2024-08-01, 2024-09-01",Problème : bug — situation : blocage_parcours


In [19]:
out_path = OUT_DIR / "block5_backlog_prioritized.csv"
backlog.to_csv(out_path, index=False)

print("✅ Backlog exporté :", out_path.resolve())
print("Nb lignes :", len(backlog))


✅ Backlog exporté : C:\Users\Malaka\Desktop\insightia_nlp\outputs\block5_backlog_prioritized.csv
Nb lignes : 21


In [20]:
backlog[[
    "problem_statement",
    col_volume,
    "priority_score"
]].head(20)


,problem_statement,n_recent,priority_score
10,Problème : photo — situation : blocage_parcours,267,261.153285
5,Problème : photo — situation : perte_temps,168,183.174194
16,Problème : photo — situation : essais_multiples,137,118.162500
1,Problème : parcours — situation : blocage_parc...,91,116.277778
12,Problème : parcours — situation : perte_temps,109,101.610169
13,Problème : parcours — situation : essais_multi...,112,101.248000
8,Problème : lenteur — situation : perte_temps,73,74.000000
0,Problème : bug — situation : essais_multiples,49,66.216216
9,Problème : lenteur — situation : essais_multiples,66,66.000000
18,Problème : bug — situation : blocage_parcours,66,55.275000
